In [28]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
#from cuml.feature_extraction.text import TfidfVectorizer
import numpy as np

In [29]:
from collections import Counter


In [4]:
import spacy
nlp = spacy.load("en_core_web_lg")

/home/jupyter/miniconda3/envs/jupyter-lab/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [30]:
input_file = "/data/elugos/event_embedding/train_20251216162745.csv"


df = pd.read_csv(input_file)

df.columns

/tmp/ipykernel_3082198/3586366695.py:4: DtypeWarning: Columns (40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file)


Index(['index', 'GlobalEventID', 'date', 'Year', 'Actor1Code', 'Actor1Name',
       'Actor1CountryCode', 'Actor1EthnicCode', 'Actor2Code', 'Actor2Name',
       'Actor2CountryCode', 'Actor2EthnicCode', 'IsRootEvent', 'EventCode',
       'EventBaseCode', 'EventRootCode', 'QuadClass', 'GoldsteinScale',
       'NumMentions', 'NumSources', 'NumArticles', 'AvgTone',
       'Actor1Geo_CountryCode', 'Actor2Geo_CountryCode', 'ActionGeo_Type',
       'ActionGeo_Fullname', 'ActionGeo_CountryCode', 'ActionGeo_ADM1Code',
       'ActionGeo_Lat', 'ActionGeo_Long', 'ActionGeo_FeatureID', 'DATEADDED',
       'SOURCEURL', 'domain', 'target', 'title', 'text', 'description',
       'sbert_text_title', 'embedding_json_title', 'sbert_text',
       'embedding_json', 'first_para', 'first_para_emb', 'PloverCode',
       'EventLabel'],
      dtype='object')

In [31]:
drop_file = "/data/elugos/event_embedding/train_202512161628_drop_dupe_titles.csv"

df_drop = pd.read_csv(drop_file)

/tmp/ipykernel_3082198/1228214834.py:3: DtypeWarning: Columns (40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  df_drop = pd.read_csv(drop_file)


In [6]:
tfidf = TfidfVectorizer(
    dtype=np.float32,
    stop_words='english',
    max_df=0.7,
    max_features=50
)
#x_tfidf = tfidf.fit_transform(df['title']).toarray().get()

In [32]:
#all catergories
dfs = {k: v for k, v in df.groupby("EventLabel")}


In [33]:
dfs_drop = {k: v for k, v in df_drop.groupby("EventLabel")}

In [34]:
for i in dfs_drop:
    print(i)

AGREE
AID
ASSAULT
COERCE
CONCEDE
CONSULT
COOPERATE
DEMAND
DISAPPROVE
MOBILIZE
PROTEST
REJECT
RETREAT
SANCTION
SUPPORT
THREATHEN


In [35]:
def add_top_words_per_group(dfs, text_col="text", top_k=5):
    out = {}

    for key, sub_df in dfs.items():
        sub_df = sub_df.copy()

        # Handle empty or all-null text safely
        texts = sub_df[text_col].fillna("").astype(str)

        tfidf = TfidfVectorizer(
            dtype=np.float32,
            stop_words="english",
            max_df=0.7,
            max_features=50
        )

        X = tfidf.fit_transform(texts)
        feature_names = np.array(tfidf.get_feature_names_out())

        # Extract top words per row
        top_words = []
        for row in X:
            if row.nnz == 0:
                top_words.append([])
            else:
                indices = row.toarray().ravel().argsort()[-top_k:][::-1]
                top_words.append(feature_names[indices].tolist())

        sub_df["top_words"] = top_words
        out[key] = sub_df

    return out

In [36]:
dfs_with_tfidf = add_top_words_per_group(dfs,top_k=20)

In [42]:
dfs_with_tfidf['AID']

,index,GlobalEventID,date,Year,Actor1Code,Actor1Name,Actor1CountryCode,Actor1EthnicCode,Actor2Code,Actor2Name,...,description,sbert_text_title,embedding_json_title,sbert_text,embedding_json,first_para,first_para_emb,PloverCode,EventLabel,top_words
12,16,1200205786,20240925,2024,USA,FLORIDA,USA,NaN,NaN,NaN,...,Tropical Storm Helene's trajectory threatens t...,"Tropical Storm Helene threatens the U.S., Mexico","[0.007524872664362192, 0.10208725184202194, 0....",NaN,NaN,Tropical storm and hurricane watches were issu...,"[0.008987472392618656, 0.0025142861995846033, ...",AID,AID,"[florida, water, told, national, just, state, ..."
83,107,1200222653,20240925,2024,USA,FLORIDA,USA,NaN,USAGOV,UNITED STATES,...,Ron DeSantis’ administration is funding ads de...,DeSantis Spends Taxpayer Money to Warn About '...,"[0.04595786705613136, 0.09839852899312973, -0....",NaN,NaN,Conservative Florida Gov. Ron DeSantis is pull...,"[-0.003946694079786539, 0.0418684221804142, -0...",AID,AID,"[florida, health, state, says, department, pub..."
84,108,1200222707,20240925,2024,USAMED,FLORIDA,USA,NaN,GOV,ADMINISTRATION,...,Ron DeSantis’ administration is funding ads de...,DeSantis Spends Taxpayer Money to Warn About '...,"[0.04595786705613136, 0.09839852899312973, -0....",NaN,NaN,Conservative Florida Gov. Ron DeSantis is pull...,"[-0.003946694079786539, 0.0418684221804142, -0...",AID,AID,"[florida, health, state, says, department, pub..."
110,155,1200236068,20240925,2024,USAGOV,NASA,USA,NaN,NaN,NaN,...,NASA-SpaceX Crew 9 mission has now been schedu...,"NASA, SpaceX delay Crew-9 mission tasked to br...","[-0.04266781359910965, 0.01877169869840145, 0....",NaN,NaN,"NASA’s SpaceX Crew-9 mission, which has been t...","[-0.036593664437532425, -0.057864680886268616,...",AID,AID,"[florida, said, program, day, help, just, time..."
112,157,1200236082,20240925,2024,USAGOV,NASA,USA,NaN,USA,UNITED STATES,...,NASA-SpaceX Crew 9 mission has now been schedu...,"NASA, SpaceX delay Crew-9 mission tasked to br...","[-0.04266781359910965, 0.01877169869840145, 0....",NaN,NaN,"NASA’s SpaceX Crew-9 mission, which has been t...","[-0.036593664437532425, -0.057864680886268616,...",AID,AID,"[florida, said, program, day, help, just, time..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89856,122790,679642700,20170808,2017,USA,VIRGINIA,USA,NaN,NaN,NaN,...,"Avizia, a leading provider of system-wide tele...",Avizia to expand healthcare IT operation in Fa...,"[0.062479496002197266, 0.04440399631857872, -0...",NaN,NaN,"Avizia, a leading provider of system-wide tele...","[0.043314699083566666, -0.05537300184369087, -...",AID,AID,"[county, company, new, said, like, health, wor..."
89858,122792,679642839,20170808,2017,USA,UNITED STATES,USA,NaN,EDU,STUDENT,...,Kelly Shushok creates an environment for self-...,Interfaith consultant analyzes spiritual cultu...,"[0.03528764471411705, 0.03554699942469597, -0....",NaN,NaN,Shushok’s commitment to the Blacksburg communi...,"[-0.02673555538058281, 0.03598686307668686, -0...",AID,AID,"[people, work, said, help, years, united, comm..."
89870,122805,679647647,20170808,2017,USA,UNITED STATES,USA,NaN,NaN,NaN,...,NaN,Business & Practice,"[0.012175978161394596, 0.03666186332702637, -0...",NaN,NaN,I’ve taken both personal delight and dismay at...,"[-0.06365057826042175, 0.045531973242759705, 0...",AID,AID,"[million, world, government, public, including..."
89871,122806,679647649,20170808,2017,USA,CALIFORNIA,USA,NaN,NaN,NaN,...,This is just insane… it’s national suicide if ...,Left Claiming 'Apartheid' If Non-citizens Are ...,"[0.007430342957377434, 0.07771247625350952, -0...",NaN,NaN,Left Claiming ‘Apartheid’ If Non-citizens Are ...,"[0.0047860643826425076, 0.07197187840938568, -...",AID,AID,"[california, law, state, federal, american, go..."


In [75]:
drops_w_tfidf = add_top_words_per_group(dfs_drop,top_k=20)

In [41]:
drops_w_tfidf['AID']

NameError: name 'drops_w_tfidf' is not defined

In [38]:
from collections import Counter

word_counts = {}

for key, sub_df in dfs_with_tfidf.items():
    counter = Counter()

    for words in sub_df["top_words"]:
        counter.update(words)   # words is a list

    word_counts[key] = dict(counter.most_common())


In [39]:
drop_word_counts = {}

for key, sub_df in drops_w_tfidf.items():
    counter = Counter()

    for words in sub_df["top_words"]:
        counter.update(words)   # words is a list

    drop_word_counts[key] = dict(counter.most_common())

NameError: name 'drops_w_tfidf' is not defined

In [40]:
for k,v in word_counts.items():
    print(k,v)

AGREE {'said': 5634, 'york': 4964, 'years': 4881, 'says': 4869, 'year': 4577, 'work': 4194, 'world': 4083, 'way': 4002, 'new': 3966, 'white': 3912, 'trump': 3856, 'told': 3852, 'time': 3795, 'school': 3765, 'united': 3575, 'state': 3111, 'support': 2618, 'people': 2599, 'think': 2508, 'states': 2421, 'president': 2201, 'like': 2070, 'just': 1978, 'including': 1725, 'according': 1626, 'city': 1614, 'day': 1592, 'make': 1582, 'right': 1557, 'news': 1502, 'help': 1492, 'public': 1446, 'going': 1440, '000': 1430, 'did': 1411, 'know': 1364, 'american': 1288, 'don': 1256, 'national': 1231, 'law': 1218, 'department': 1186, 'house': 1169, 'million': 1163, 'government': 1136, 'company': 1030, 'florida': 1000, 'california': 994, 'police': 889, 'court': 855, 'attack': 753}
AID {'public': 3392, 'right': 3342, 'said': 3336, 'york': 3312, 'years': 3184, 'year': 3069, 'work': 2729, 'time': 2622, 'new': 2579, 'world': 2547, 'united': 2536, 'trump': 2485, 'water': 2323, 'state': 2318, 'told': 2308, 'su

In [78]:
for k,v in drop_word_counts.items():
    print(k,v)

AGREE {'said': 2544, 'public': 2267, 'york': 2260, 'years': 2241, 'year': 2116, 'way': 1974, 'time': 1962, 'work': 1929, 'world': 1874, 'says': 1809, 'united': 1799, 'new': 1777, 'trump': 1777, 'told': 1707, 'state': 1328, 'think': 1285, 'support': 1237, 'states': 1170, 'people': 1156, 'president': 1019, 'school': 998, 'like': 938, 'just': 884, 'including': 778, 'make': 757, 'city': 725, 'day': 705, 'according': 697, 'help': 675, 'news': 673, 'going': 646, '000': 633, 'know': 617, 'need': 612, 'did': 606, 'long': 603, 'american': 564, 'national': 548, 'don': 543, 'million': 525, 'florida': 518, 'house': 507, 'community': 502, 'law': 502, 'company': 487, 'government': 481, 'federal': 466, 'california': 445, 'police': 389, 'health': 345}
AID {'president': 1651, 'public': 1638, 'york': 1577, 'years': 1546, 'year': 1506, 'said': 1470, 'way': 1310, 'work': 1294, 'time': 1284, 'united': 1243, 'new': 1188, 'trump': 1184, 'water': 1146, 'told': 1113, 'state': 1072, 'right': 1041, 'people': 102

In [ ]:
remove_words = ['trump','york','years','year','president']
remove_set = set(remove_words)

filtered_word_counts = {
    key: {word: count
          for word, count in counts.items()
          if word not in remove_set}
    for key, counts in word_counts.items()
}


In [79]:
drop_filtered_word_counts = {
    key: {word: count
          for word, count in counts.items()
          if word not in remove_set}
    for key, counts in drop_word_counts.items()
}


In [36]:
for k,v in filtered_word_counts.items():
    print(k,v)

AGREE {'said': 5634, 'says': 4869, 'work': 4194, 'world': 4083, 'way': 4002, 'new': 3966, 'white': 3912, 'told': 3852, 'time': 3795, 'school': 3765, 'united': 3575, 'state': 3111, 'support': 2618, 'people': 2599, 'think': 2508, 'states': 2421, 'like': 2070, 'just': 1978, 'including': 1725, 'according': 1626, 'city': 1614, 'day': 1592, 'make': 1582, 'right': 1557, 'news': 1502, 'help': 1492, 'public': 1446, 'going': 1440, '000': 1430, 'did': 1411, 'know': 1364, 'american': 1288, 'don': 1256, 'national': 1231, 'law': 1218, 'department': 1186, 'house': 1169, 'million': 1163, 'government': 1136, 'company': 1030, 'florida': 1000, 'california': 994, 'police': 889, 'court': 855, 'attack': 753}
AID {'public': 3392, 'right': 3342, 'said': 3336, 'work': 2729, 'time': 2622, 'new': 2579, 'world': 2547, 'united': 2536, 'water': 2323, 'state': 2318, 'told': 2308, 'support': 2099, 'people': 2039, 'states': 1896, 'says': 1604, 'just': 1427, 'like': 1426, 'including': 1391, 'help': 1351, 'according': 1

In [80]:
for k,v in drop_filtered_word_counts.items():
    print(k,v)

AGREE {'said': 2544, 'public': 2267, 'way': 1974, 'time': 1962, 'work': 1929, 'world': 1874, 'says': 1809, 'united': 1799, 'new': 1777, 'told': 1707, 'state': 1328, 'think': 1285, 'support': 1237, 'states': 1170, 'people': 1156, 'school': 998, 'like': 938, 'just': 884, 'including': 778, 'make': 757, 'city': 725, 'day': 705, 'according': 697, 'help': 675, 'news': 673, 'going': 646, '000': 633, 'know': 617, 'need': 612, 'did': 606, 'long': 603, 'american': 564, 'national': 548, 'don': 543, 'million': 525, 'florida': 518, 'house': 507, 'community': 502, 'law': 502, 'company': 487, 'government': 481, 'federal': 466, 'california': 445, 'police': 389, 'health': 345}
AID {'public': 1638, 'said': 1470, 'way': 1310, 'work': 1294, 'time': 1284, 'united': 1243, 'new': 1188, 'water': 1146, 'told': 1113, 'state': 1072, 'right': 1041, 'people': 1027, 'support': 1022, 'states': 938, 'just': 657, 'like': 652, 'including': 615, 'help': 614, 'according': 586, '000': 565, 'school': 560, 'million': 541, '

In [43]:
def add_empath_per_group(dfs, text_col="text", normalize=True):
    out = {}

    for key, sub_df in dfs.items():
        sub_df = sub_df.copy()

        texts = sub_df[text_col].fillna("").astype(str)

        empath_scores = []
        for t in texts:
            if t.strip():
                empath_scores.append(
                    lexicon.analyze(t, normalize=normalize)
                )
            else:
                empath_scores.append({})

        sub_df["empath"] = empath_scores
        out[key] = sub_df

    return out

In [42]:
empath_df = add_empath_per_group(dfs)

In [81]:
drop_empath_df = add_empath_per_group(dfs_drop)

In [83]:
drop_empath_df['AID']

,index,GlobalEventID,date,Year,Actor1Code,Actor1Name,Actor1CountryCode,Actor1EthnicCode,Actor2Code,Actor2Name,...,description,sbert_text_title,embedding_json_title,sbert_text,embedding_json,first_para,first_para_emb,PloverCode,EventLabel,empath
6,16,1200205786,20240925,2024,USA,FLORIDA,USA,NaN,NaN,NaN,...,Tropical Storm Helene's trajectory threatens t...,"Tropical Storm Helene threatens the U.S., Mexico","[0.007524872664362192, 0.10208725184202194, 0....",NaN,NaN,Tropical storm and hurricane watches were issu...,"[0.008987472392618656, 0.0025142861995846033, ...",AID,AID,"{'help': 0.0, 'office': 0.0, 'dance': 0.0, 'mo..."
53,192,1200246221,20240925,2024,MIL,CARRIER,NaN,NaN,NaN,NaN,...,Florida’s Third District Court of Appeal recen...,Order Denying Motion to Dismiss under Florida’...,"[-0.008244303055107594, 0.02947995625436306, 0...",NaN,NaN,Florida’s Third District Court of Appeal recen...,"[0.008134478703141212, -0.002518073422834277, ...",AID,AID,"{'help': 0.006557377049180328, 'office': 0.0, ..."
55,194,1200247990,20240925,2024,REL,ABBOT,NaN,NaN,USA,FLORIDA,...,Tropical Storm Helene is expected to strengthe...,Gov. Abbott sends search and rescue team to Fl...,"[-0.02673930488526821, -0.04822605848312378, 0...",NaN,NaN,Tropical Storm Helene is expected to strengthe...,"[0.0016634989297017455, 0.002456099260598421, ...",AID,AID,"{'help': 0.014925373134328358, 'office': 0.0, ..."
56,195,1200248221,20240925,2024,USA,UNITED STATES,USA,NaN,HLH,MEDICAL SPECIALIST,...,The team headed to Florida is made up of 80 me...,'It's a calling' | 9 women among those on VA T...,"[-0.02080177329480648, 0.003331208834424615, 0...",NaN,NaN,The team headed to Florida is made up of 80 me...,"[0.023279637098312378, -0.04088704660534859, -...",AID,AID,"{'help': 0.01366742596810934, 'office': 0.0022..."
108,351,1200326583,20240925,2024,USA,FLORIDA,USA,NaN,NaN,NaN,...,House approves bill to study making US Jewish ...,Museum on US Jewish history to become part of ...,"[0.008966092951595783, 0.09074261039495468, -0...",NaN,NaN,American Jews are one step closer to having a ...,"[0.04561415687203407, 0.10061408579349518, -0....",AID,AID,"{'help': 0.00683371298405467, 'office': 0.0, '..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37833,122714,679572657,20170808,2017,MED,WEBSITE,NaN,NaN,CANEDU,MCGILL,...,From Pathocracy to the (Mis)Anthropocene Psych...,Ecocide and the Psychotic 0.5 Per Cent,"[0.06396777927875519, 0.04475044086575508, -0....",NaN,NaN,From Pathocracy to the (Mis)Anthropocene,"[-0.0013103554956614971, 0.01027937326580286, ...",AID,AID,"{'help': 0.002331002331002331, 'office': 0.001..."
37849,122772,679630956,20170808,2017,BUS,AEROSPACE,NaN,NaN,NaN,NaN,...,"Leesburg, VA (PRWEB) August 08, 2017 -- ProJet...",ProJet Aviation promotes Julie O’Brien and Tin...,"[0.01783275231719017, -0.0002711133856792003, ...",NaN,NaN,"Leesburg, VA (PRWEB) August 08, 2017 -- ProJet...","[0.03776798024773598, 0.04905354231595993, -0....",AID,AID,"{'help': 0.011647254575707155, 'office': 0.006..."
37854,122790,679642700,20170808,2017,USA,VIRGINIA,USA,NaN,NaN,NaN,...,"Avizia, a leading provider of system-wide tele...",Avizia to expand healthcare IT operation in Fa...,"[0.062479496002197266, 0.04440399631857872, -0...",NaN,NaN,"Avizia, a leading provider of system-wide tele...","[0.043314699083566666, -0.05537300184369087, -...",AID,AID,"{'help': 0.009881422924901186, 'office': 0.001..."
37855,122792,679642839,20170808,2017,USA,UNITED STATES,USA,NaN,EDU,STUDENT,...,Kelly Shushok creates an environment for self-...,Interfaith consultant analyzes spiritual cultu...,"[0.03528764471411705, 0.03554699942469597, -0....",NaN,NaN,Shushok’s commitment to the Blacksburg communi...,"[-0.02673555538058281, 0.03598686307668686, -0...",AID,AID,"{'help': 0.0029411764705882353, 'office': 0.00..."


In [47]:
def sorted_nonzero_empath(d):
    return sorted(
        ((k, v) for k, v in d.items() if v > 0),
        key=lambda x: x[1],
        reverse=True
    )




In [49]:
for key, df in empath_df.items():
    df["empath_sorted"] = df["empath"].apply(sorted_nonzero_empath)


In [84]:
for key, df in drop_empath_df.items():
    df["empath_sorted"] = df["empath"].apply(sorted_nonzero_empath)

In [51]:
group_empath = {}

for key, df in empath_df.items():
    agg = Counter()
    for d in df["empath"]:
        agg.update(d)

    group_empath[key] = dict(agg.most_common())

In [85]:
drop_group_empath = {}

for key, df in drop_empath_df.items():
    agg = Counter()
    for d in df["empath"]:
        agg.update(d)

    drop_group_empath[key] = dict(agg.most_common())

In [52]:
group_empath['AID']

{'business': 43.56922197351027,
 'government': 36.886458114447,
 'economics': 31.87855262729575,
 'work': 29.240785367591688,
 'money': 28.58200639209145,
 'giving': 27.192476208568735,
 'law': 26.38987483147518,
 'help': 25.615626912516102,
 'leader': 24.269268441979122,
 'communication': 23.586954454561543,
 'payment': 22.773856995748755,
 'valuable': 22.69232368974326,
 'banking': 22.55997886333286,
 'college': 21.263980347765557,
 'internet': 20.300075390312067,
 'crime': 20.18001674398241,
 'school': 18.92523697708976,
 'meeting': 17.877785075252845,
 'real_estate': 17.190712549915457,
 'negative_emotion': 16.84796337140489,
 'technology': 16.62294111884675,
 'politics': 16.483960457165864,
 'speaking': 16.040573931342198,
 'wealthy': 15.689415497672552,
 'messaging': 15.465649364494919,
 'office': 15.418338440292963,
 'dispute': 14.516113324021825,
 'social_media': 13.676822815806826,
 'gain': 13.378690796041244,
 'celebration': 13.242698937610038,
 'stealing': 13.179280974146323

In [86]:
for k,v in drop_group_empath.items():
    print(k,v)

AGREE {'business': 26.278045405563798, 'government': 17.011676206440143, 'work': 16.279847153798165, 'economics': 16.0839226233017, 'communication': 15.482275572838837, 'meeting': 14.556895893237016, 'leader': 14.454158079059148, 'giving': 14.323298483642887, 'money': 13.154306198293725, 'law': 13.004636363044694, 'crime': 12.432843370523106, 'help': 12.225416178638916, 'party': 12.067150414060876, 'school': 11.750528167465594, 'speaking': 11.637187584696331, 'celebration': 11.611061686638275, 'college': 11.512110161939558, 'traveling': 11.439273828315747, 'internet': 11.254434568459148, 'office': 10.665817091749664, 'valuable': 10.463180211131993, 'payment': 10.373808660857842, 'real_estate': 10.367234502144271, 'vacation': 10.332347523908266, 'banking': 10.199332470213497, 'negative_emotion': 10.160035974207316, 'messaging': 9.52425902814506, 'technology': 9.295034007486969, 'politics': 8.948988804553919, 'dispute': 8.763793818007581, 'family': 8.636840633234259, 'social_media': 8.37

In [54]:
for k,v in group_empath.items():
    print(k,v)

AGREE {'business': 56.511528167407164, 'government': 39.236374858030324, 'work': 36.16088514785117, 'communication': 34.90537559913113, 'meeting': 33.99005315915082, 'economics': 33.76819457662186, 'leader': 33.678622609892656, 'law': 31.264163881414706, 'giving': 31.093997563989173, 'crime': 29.763700859783444, 'money': 28.26543515904663, 'help': 26.6462784169926, 'speaking': 26.140805894844576, 'traveling': 25.949831423928302, 'school': 25.92436422694717, 'college': 25.608435104770756, 'party': 25.013044290076387, 'office': 24.98778814150994, 'internet': 24.837884394059, 'celebration': 24.68655388869227, 'vacation': 22.694238917284974, 'banking': 22.59423239225709, 'negative_emotion': 22.473734311757237, 'payment': 22.141053462984903, 'dispute': 22.102005082692816, 'valuable': 21.94248568019802, 'messaging': 21.319348570914293, 'politics': 20.79875726106351, 'stealing': 19.96132067082979, 'technology': 19.78362107587388, 'real_estate': 19.728942402563803, 'family': 19.240580266365257

In [5]:
df = df.copy()

df_dedup = (
    df.drop_duplicates(
        subset=["title", "EventLabel"],
        keep="first"
    ))

In [6]:
df_dedup

,index,GlobalEventID,date,Year,Actor1Code,Actor1Name,Actor1CountryCode,Actor1EthnicCode,Actor2Code,Actor2Name,...,text,description,sbert_text_title,embedding_json_title,sbert_text,embedding_json,first_para,first_para_emb,PloverCode,EventLabel
0,0,1200203233,20240918,2024,USA,UNITED STATES,USA,NaN,CRM,GANG,...,MIAMI (AP) — Former federal agent Was Tabor sa...,The gang has exploded into the presidential ca...,Tren de Aragua gang started in Venezuela’s pri...,"[-0.002818030072376132, -0.01742059737443924, ...",NaN,NaN,MIAMI (AP) — Former federal agent Was Tabor sa...,"[-0.08164525777101517, -0.027655666694045067, ...",ASSAULT,ASSAULT
1,2,1200203284,20240925,2024,NaN,NaN,NaN,NaN,CHR,CHRISTIAN,...,NASHVILLE (BP) — The word “tumultuous” is one ...,NASHVILLE (BP) — The word “tumultuous” is one ...,"Yarnell: In times of competing allegiances, re...","[-0.0645490437746048, 0.05730828642845154, 0.0...",NaN,NaN,NASHVILLE (BP) — The word “tumultuous” is one ...,"[0.017953015863895416, -0.06336908042430878, -...",DEMAND,DEMAND
2,3,1200203520,20240925,2024,NaN,NaN,NaN,NaN,USA,FLORIDA,...,Yahoo is using AI to generate takeaways from t...,Weather experts are expecting Hurricane Helene...,How Strong Could Hurricane Helene Get When It ...,"[-0.018027156591415405, 0.024802444502711296, ...",NaN,NaN,Yahoo is using AI to generate takeaways from t...,"[-0.018082909286022186, 0.06462761014699936, -...",CONSULT,CONSULT
4,5,1200203590,20240925,2024,NaN,NaN,NaN,NaN,USA,FLORIDA,...,Yahoo is using AI to generate takeaways from t...,Since the non-endorsement by the national gove...,"Florida, Georgia Teamsters break with national...","[-0.049125298857688904, -0.03338903561234474, ...",NaN,NaN,Yahoo is using AI to generate takeaways from t...,"[-0.018082909286022186, 0.06462761014699936, -...",ASSAULT,ASSAULT
5,7,1200204067,20240925,2024,CRM,GANG,NaN,NaN,USA,UNITED STATES,...,MIAMI (AP) — Former federal agent Was Tabor sa...,The gang has exploded into the presidential ca...,Tren de Aragua gang started in Venezuela’s pri...,"[-0.002818030072376132, -0.01742059737443924, ...",NaN,NaN,MIAMI (AP) — Former federal agent Was Tabor sa...,"[-0.08164525777101517, -0.027655666694045067, ...",REJECT,REJECT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89982,122970,679730864,20170808,2017,USA,UNITED STATES,USA,NaN,GOV,PRESIDENT,...,President Donald Trump promised an America-Fir...,President Donald Trump promised an America-Fir...,"Trump, Cooper hold strong positions on Atlanti...","[0.001870636478997767, -0.035346224904060364, ...",NaN,NaN,President Donald Trump promised an America-Fir...,"[-0.07812009751796722, 0.0563594251871109, 0.0...",DISAPPROVE,DISAPPROVE
89985,122973,679732457,20170808,2017,EDU,SCHOOL,NaN,NaN,NaN,NaN,...,President Donald Trump was briefed today about...,President Donald Trump was briefed today about...,WVU Providing Multi-Pronged Approach Solving t...,"[0.015047898516058922, -0.004980273544788361, ...",NaN,NaN,President Donald Trump was briefed today about...,"[0.0483584962785244, -0.012323571369051933, 0....",COOPERATE,COOPERATE
89986,122975,679733002,20170808,2017,MED,BLOGGER,NaN,NaN,BUS,INDUSTRY,...,Who's the Guru? It's you – local voters in the...,NaN,Loudoun County to host 2018 beer bloggers conf...,"[0.020191552117466927, -0.014680474996566772, ...",NaN,NaN,Who's the Guru? It's you – local voters in the...,"[0.028580108657479286, -0.04236813634634018, 0...",CONSULT,CONSULT
89987,122977,679733469,20170808,2017,USA,UNITED STATES,USA,NaN,NaN,NaN,...,Ed Gillespie and Ralph Northam say they will d...,Too many of Virginia's schools are in dire shape.,Opinion | Bricks and mortar may be key in the ...,"[0.03240766003727913, 0.12859293818473816, 0.0...",NaN,NaN,Ed Gillespie and Ralph Northam say they will d...,"[-0.003782308427616954, 0.0422658771276474, 0....",RETREAT,RETREAT


In [7]:
df_ec = df.copy()

df_ec["event_label_concat"] = (
    df_ec.groupby("title")["EventLabel"]
      .transform(lambda x: "_".join(sorted(set(x.astype(str)))))
)

In [8]:
df_final = (
    df_ec.drop_duplicates(subset=["title"])
      .reset_index(drop=True)
)

In [9]:
df_final

,index,GlobalEventID,date,Year,Actor1Code,Actor1Name,Actor1CountryCode,Actor1EthnicCode,Actor2Code,Actor2Name,...,description,sbert_text_title,embedding_json_title,sbert_text,embedding_json,first_para,first_para_emb,PloverCode,EventLabel,event_label_concat
0,0,1200203233,20240918,2024,USA,UNITED STATES,USA,NaN,CRM,GANG,...,The gang has exploded into the presidential ca...,Tren de Aragua gang started in Venezuela’s pri...,"[-0.002818030072376132, -0.01742059737443924, ...",NaN,NaN,MIAMI (AP) — Former federal agent Was Tabor sa...,"[-0.08164525777101517, -0.027655666694045067, ...",ASSAULT,ASSAULT,ASSAULT_COERCE_REJECT
1,2,1200203284,20240925,2024,NaN,NaN,NaN,NaN,CHR,CHRISTIAN,...,NASHVILLE (BP) — The word “tumultuous” is one ...,"Yarnell: In times of competing allegiances, re...","[-0.0645490437746048, 0.05730828642845154, 0.0...",NaN,NaN,NASHVILLE (BP) — The word “tumultuous” is one ...,"[0.017953015863895416, -0.06336908042430878, -...",DEMAND,DEMAND,DEMAND
2,3,1200203520,20240925,2024,NaN,NaN,NaN,NaN,USA,FLORIDA,...,Weather experts are expecting Hurricane Helene...,How Strong Could Hurricane Helene Get When It ...,"[-0.018027156591415405, 0.024802444502711296, ...",NaN,NaN,Yahoo is using AI to generate takeaways from t...,"[-0.018082909286022186, 0.06462761014699936, -...",CONSULT,CONSULT,AGREE_CONSULT
3,5,1200203590,20240925,2024,NaN,NaN,NaN,NaN,USA,FLORIDA,...,Since the non-endorsement by the national gove...,"Florida, Georgia Teamsters break with national...","[-0.049125298857688904, -0.03338903561234474, ...",NaN,NaN,Yahoo is using AI to generate takeaways from t...,"[-0.018082909286022186, 0.06462761014699936, -...",ASSAULT,ASSAULT,ASSAULT_SUPPORT_THREATHEN
4,9,1200204213,20240925,2024,EDU,STUDENT,NaN,NaN,NaN,NaN,...,Read how members of the university community r...,A day to remember: FIU celebrates Top 50,"[0.021655291318893433, -0.011736852116882801, ...",NaN,NaN,The university that never stops has experience...,"[0.07788405567407608, -0.07917836308479309, 0....",CONSULT,CONSULT,CONSULT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37908,122970,679730864,20170808,2017,USA,UNITED STATES,USA,NaN,GOV,PRESIDENT,...,President Donald Trump promised an America-Fir...,"Trump, Cooper hold strong positions on Atlanti...","[0.001870636478997767, -0.035346224904060364, ...",NaN,NaN,President Donald Trump promised an America-Fir...,"[-0.07812009751796722, 0.0563594251871109, 0.0...",DISAPPROVE,DISAPPROVE,DISAPPROVE
37909,122973,679732457,20170808,2017,EDU,SCHOOL,NaN,NaN,NaN,NaN,...,President Donald Trump was briefed today about...,WVU Providing Multi-Pronged Approach Solving t...,"[0.015047898516058922, -0.004980273544788361, ...",NaN,NaN,President Donald Trump was briefed today about...,"[0.0483584962785244, -0.012323571369051933, 0....",COOPERATE,COOPERATE,COOPERATE
37910,122975,679733002,20170808,2017,MED,BLOGGER,NaN,NaN,BUS,INDUSTRY,...,NaN,Loudoun County to host 2018 beer bloggers conf...,"[0.020191552117466927, -0.014680474996566772, ...",NaN,NaN,Who's the Guru? It's you – local voters in the...,"[0.028580108657479286, -0.04236813634634018, 0...",CONSULT,CONSULT,CONSULT
37911,122977,679733469,20170808,2017,USA,UNITED STATES,USA,NaN,NaN,NaN,...,Too many of Virginia's schools are in dire shape.,Opinion | Bricks and mortar may be key in the ...,"[0.03240766003727913, 0.12859293818473816, 0.0...",NaN,NaN,Ed Gillespie and Ralph Northam say they will d...,"[-0.003782308427616954, 0.0422658771276474, 0....",RETREAT,RETREAT,RETREAT


In [10]:
df_multiple_codes = df_final[
    df_final["event_label_concat"].str.contains("_")  # fallback heuristic
]

In [11]:
df_multiple_codes

,index,GlobalEventID,date,Year,Actor1Code,Actor1Name,Actor1CountryCode,Actor1EthnicCode,Actor2Code,Actor2Name,...,description,sbert_text_title,embedding_json_title,sbert_text,embedding_json,first_para,first_para_emb,PloverCode,EventLabel,event_label_concat
0,0,1200203233,20240918,2024,USA,UNITED STATES,USA,NaN,CRM,GANG,...,The gang has exploded into the presidential ca...,Tren de Aragua gang started in Venezuela’s pri...,"[-0.002818030072376132, -0.01742059737443924, ...",NaN,NaN,MIAMI (AP) — Former federal agent Was Tabor sa...,"[-0.08164525777101517, -0.027655666694045067, ...",ASSAULT,ASSAULT,ASSAULT_COERCE_REJECT
2,3,1200203520,20240925,2024,NaN,NaN,NaN,NaN,USA,FLORIDA,...,Weather experts are expecting Hurricane Helene...,How Strong Could Hurricane Helene Get When It ...,"[-0.018027156591415405, 0.024802444502711296, ...",NaN,NaN,Yahoo is using AI to generate takeaways from t...,"[-0.018082909286022186, 0.06462761014699936, -...",CONSULT,CONSULT,AGREE_CONSULT
3,5,1200203590,20240925,2024,NaN,NaN,NaN,NaN,USA,FLORIDA,...,Since the non-endorsement by the national gove...,"Florida, Georgia Teamsters break with national...","[-0.049125298857688904, -0.03338903561234474, ...",NaN,NaN,Yahoo is using AI to generate takeaways from t...,"[-0.018082909286022186, 0.06462761014699936, -...",ASSAULT,ASSAULT,ASSAULT_SUPPORT_THREATHEN
7,19,1200205864,20240925,2024,USA,CHARLOTTE,USA,NaN,NaN,NaN,...,"One county warned that the storm, which is exp...",Tropical Storm Helene evacuation map as storm ...,"[0.0638221725821495, 0.08435751497745514, 0.04...",NaN,NaN,Several Florida counties have issued mandatory...,"[0.030011611059308052, -0.024009326472878456, ...",SANCTION,SANCTION,SANCTION_SUPPORT
8,20,1200205887,20240925,2024,USA,UNITED STATES,USA,NaN,NaN,NaN,...,"Ryan Routh, the suspect in the second Trump as...",Ryan Routh charged with attempted assassinatio...,"[-0.03664088249206543, 0.05994853749871254, -0...",NaN,NaN,"Ryan Routh, the suspect in the second Trump as...","[-0.008147599175572395, 0.021718882024288177, ...",ASSAULT,ASSAULT,ASSAULT_COERCE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37861,122823,679656651,20170808,2017,MIL,MILITARY BASE,NaN,NaN,NaN,NaN,...,The armed forces are concerned about more than...,Here's Why the U.S. Military is Scared of Drones,"[0.07015137374401093, 0.012341306544840336, 0....",NaN,NaN,"Among the dangers faced by the U.S. military, ...","[0.10064809769392014, -0.015102533623576164, -...",ASSAULT,ASSAULT,ASSAULT_CONSULT
37880,122877,679687790,20170808,2017,USACVL,UNITED STATES,USA,NaN,NaN,NaN,...,Can a Republican win there?,Opinion | The GOP’s challenge in Virginia,"[-0.0002878196828532964, 0.03428173065185547, ...",NaN,NaN,In the latest poll in Virginia’s gubernatorial...,"[0.06598247587680817, -0.015836024656891823, -...",DISAPPROVE,DISAPPROVE,DISAPPROVE_REJECT
37885,122887,679695385,20170808,2017,USA,VIRGINIA,USA,NaN,BUS,BUSINESS,...,"Draftco Inc., a machine and fabrication shop, ...","Draftco Inc. to invest $450,000, create 16 new...","[-0.013736411929130554, -0.023327047005295753,...",NaN,NaN,"Draftco Inc., a machine and fabrication shop, ...","[-0.03974677994847298, 0.006161803845316172, -...",SUPPORT,SUPPORT,AID_SUPPORT
37891,122902,679700492,20170808,2017,NaN,NaN,NaN,NaN,MIL,ARMY,...,NaN,Defense Systems,"[-0.04630216211080551, 0.04322230815887451, -0...",NaN,NaN,"News, analysis, and ideas driving the future o...","[-0.0165393128991127, -0.028577638790011406, -...",CONSULT,CONSULT,ASSAULT_CONSULT_COOPERATE


In [23]:
cat_labels = df_multiple_codes["event_label_concat"].tolist()

In [13]:
cat_labels.sort()

In [43]:
len(df)

89989

In [24]:
count_cat_labels = Counter(cat_labels).most_common()

In [25]:
count_cat_labels

[('ASSAULT_CONSULT', 485),
 ('CONSULT_SUPPORT', 394),
 ('CONSULT_DISAPPROVE', 393),
 ('ASSAULT_DISAPPROVE', 378),
 ('AGREE_CONSULT', 334),
 ('ASSAULT_COERCE', 255),
 ('COERCE_DISAPPROVE', 248),
 ('DISAPPROVE_SUPPORT', 232),
 ('COERCE_CONSULT', 185),
 ('AID_CONSULT', 181),
 ('ASSAULT_SUPPORT', 174),
 ('CONSULT_REJECT', 137),
 ('AGREE_SUPPORT', 132),
 ('DISAPPROVE_REJECT', 123),
 ('AID_DISAPPROVE', 114),
 ('AID_SUPPORT', 111),
 ('AGREE_DISAPPROVE', 110),
 ('COERCE_SUPPORT', 102),
 ('CONSULT_COOPERATE', 101),
 ('AGREE_AID', 97),
 ('AGREE_ASSAULT', 90),
 ('CONCEDE_CONSULT', 88),
 ('CONSULT_RETREAT', 84),
 ('AID_ASSAULT', 80),
 ('CONSULT_THREATHEN', 78),
 ('ASSAULT_REJECT', 77),
 ('CONCEDE_DISAPPROVE', 77),
 ('REJECT_SUPPORT', 76),
 ('COERCE_REJECT', 74),
 ('ASSAULT_THREATHEN', 71),
 ('ASSAULT_CONSULT_DISAPPROVE', 70),
 ('DISAPPROVE_THREATHEN', 68),
 ('ASSAULT_RETREAT', 66),
 ('ASSAULT_COOPERATE', 66),
 ('COERCE_RETREAT', 61),
 ('COERCE_PROTEST', 61),
 ('CONSULT_DEMAND', 59),
 ('ASSAULT_CON

In [26]:
len(count_cat_labels)

1063